<a href="https://colab.research.google.com/github/cafauzi13/ESG_SentimentAnalysis/blob/main/scripts/klasifikasi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pipeline C — Fine-tuning IndoBERT: Klasifikasi Sentimen & Tag
**ESG Sentiment Analysis — Greenwashing Detection**

Pipeline ini melakukan:
1. Load `train_set_augmented.csv` + `test_set_asli.csv`
2. Fine-tune IndoBERT untuk klasifikasi **Sentimen** (Positif/Negatif/Netral)
3. Fine-tune IndoBERT untuk klasifikasi **Tag** (Environment/Social/Governance/Finance/Investigation)
4. Evaluasi final di test set murni
5. Simpan model & hasil prediksi

> ⚠️ Pastikan Runtime → Change runtime type → **GPU (T4)**

## 0. Install & Import

In [ ]:
!pip install -q transformers datasets scikit-learn torch accelerate

In [ ]:
import pandas as pd
import numpy as np
import torch
import os
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Cek GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: Tesla T4


## 1. Mount Drive & Load Data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os

# ============================================================
# SESUAIKAN PATH INI dengan lokasi file di Drive kamu
# ============================================================
DRIVE_DIR = '/content/drive/MyDrive/ESG/'
TRAIN_PATH = DRIVE_DIR + 'train_set_augmented.csv'
TEST_PATH  = DRIVE_DIR + 'test_set_asli.csv'

# Fallback: load dari GitHub jika belum ada di Drive
GITHUB_BASE = 'https://raw.githubusercontent.com/cafauzi13/ESG_SentimentAnalysis/main/data/'

if os.path.exists(TRAIN_PATH):
    df_train = pd.read_csv(TRAIN_PATH)
    df_test  = pd.read_csv(TEST_PATH)
    print('✅ Loaded dari Drive')
else:
    print('⚠️ File tidak ditemukan di Drive, loading dari GitHub...')
    df_train = pd.read_csv(GITHUB_BASE + 'train_set_augmented.csv')
    df_test  = pd.read_csv(GITHUB_BASE + 'test_set_asli.csv')
    print('✅ Loaded dari GitHub')

# Bersihkan
TEXT_COL = 'Isi Berita Clean'
df_train = df_train.dropna(subset=[TEXT_COL, 'Sentiment', 'Tag'])
df_test  = df_test.dropna(subset=[TEXT_COL, 'Sentiment', 'Tag'])
df_train[TEXT_COL] = df_train[TEXT_COL].astype(str)
df_test[TEXT_COL]  = df_test[TEXT_COL].astype(str)

print(f'\nTrain: {len(df_train)} | Test: {len(df_test)}')
print(f'\nDistribusi Sentimen (Train):\n{df_train["Sentiment"].value_counts()}')
print(f'\nDistribusi Tag (Train):\n{df_train["Tag"].value_counts()}')

Mounted at /content/drive


NameError: name 'os' is not defined

## 2. Konfigurasi Global

In [ ]:
# ============================================================
# KONFIGURASI — ubah sesuai kebutuhan
# ============================================================
MODEL_NAME   = 'indobenchmark/indobert-base-p1'
MAX_LEN      = 256   # panjang token per artikel
BATCH_SIZE   = 8     # turunkan ke 4 jika OOM
EPOCHS       = 5
LR           = 2e-5
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
OUTPUT_DIR   = DRIVE_DIR + 'models/'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'Model: {MODEL_NAME}')
print(f'Max Length: {MAX_LEN} | Batch: {BATCH_SIZE} | Epochs: {EPOCHS}')

## 3. Dataset Class & Tokenizer

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'✅ Tokenizer loaded: {MODEL_NAME}')

class ESGDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids':      encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long)
        }

## 4. Helper: Metrics & Plot

In [ ]:
from sklearn.metrics import f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1_macro': f1_score(labels, preds, average='macro')
    }

def plot_confusion_matrix(y_true, y_pred, class_names, title):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names)
    plt.title(title)
    plt.ylabel('Aktual')
    plt.xlabel('Prediksi')
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + f'confusion_{title.replace(" ","_")}.png', dpi=150)
    plt.show()

def run_evaluation(trainer, test_dataset, le, task_name):
    print(f'\n{"="*50}')
    print(f'EVALUASI FINAL — {task_name}')
    print(f'{"="*50}')
    preds_output = trainer.predict(test_dataset)
    y_pred = np.argmax(preds_output.predictions, axis=1)
    y_true = preds_output.label_ids
    class_names = le.classes_
    print(classification_report(y_true, y_pred, target_names=class_names))
    plot_confusion_matrix(y_true, y_pred, class_names, task_name)
    return y_pred, y_true

## 5. Task A — Fine-tuning Klasifikasi Sentimen
Label: **Positif / Negatif / Netral**

In [ ]:
# ============================================================
# ENCODE LABEL SENTIMEN
# ============================================================
le_sent = LabelEncoder()
df_train['label_sent'] = le_sent.fit_transform(df_train['Sentiment'])
df_test['label_sent']  = le_sent.transform(df_test['Sentiment'])

print(f'Kelas Sentimen: {le_sent.classes_}')
print(f'Mapping: {dict(zip(le_sent.classes_, le_sent.transform(le_sent.classes_)))}')

In [ ]:
# ============================================================
# BUILD DATASET
# ============================================================
train_sent_ds = ESGDataset(
    df_train[TEXT_COL].tolist(),
    df_train['label_sent'].tolist(),
    tokenizer, MAX_LEN
)
test_sent_ds = ESGDataset(
    df_test[TEXT_COL].tolist(),
    df_test['label_sent'].tolist(),
    tokenizer, MAX_LEN
)
print(f'Train samples: {len(train_sent_ds)} | Test samples: {len(test_sent_ds)}')

In [ ]:
# ============================================================
# MODEL & TRAINER SENTIMEN
# ============================================================
NUM_LABELS_SENT = len(le_sent.classes_)
model_sent = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS_SENT
)

SENT_OUTPUT = OUTPUT_DIR + 'sentiment/'
os.makedirs(SENT_OUTPUT, exist_ok=True)

args_sent = TrainingArguments(
    output_dir=SENT_OUTPUT,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_steps=50,
    report_to='none',
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2
)

trainer_sent = Trainer(
    model=model_sent,
    args=args_sent,
    train_dataset=train_sent_ds,
    eval_dataset=test_sent_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print('🚀 Mulai fine-tuning Sentimen...')
trainer_sent.train()

In [ ]:
# ============================================================
# EVALUASI SENTIMEN
# ============================================================
y_pred_sent, y_true_sent = run_evaluation(
    trainer_sent, test_sent_ds, le_sent, 'Klasifikasi Sentimen'
)

# Simpan model terbaik
trainer_sent.save_model(SENT_OUTPUT + 'best_model/')
print(f'\n✅ Model sentimen disimpan: {SENT_OUTPUT}best_model/')

## 6. Task B — Fine-tuning Klasifikasi Tag
Label: **Environment / Social / Governance / Finance / Investigation**

In [ ]:
# ============================================================
# ENCODE LABEL TAG
# ============================================================
le_tag = LabelEncoder()
df_train['label_tag'] = le_tag.fit_transform(df_train['Tag'])
df_test['label_tag']  = le_tag.transform(df_test['Tag'])

print(f'Kelas Tag: {le_tag.classes_}')
print(f'Mapping: {dict(zip(le_tag.classes_, le_tag.transform(le_tag.classes_)))}')

In [ ]:
# ============================================================
# BUILD DATASET TAG
# ============================================================
train_tag_ds = ESGDataset(
    df_train[TEXT_COL].tolist(),
    df_train['label_tag'].tolist(),
    tokenizer, MAX_LEN
)
test_tag_ds = ESGDataset(
    df_test[TEXT_COL].tolist(),
    df_test['label_tag'].tolist(),
    tokenizer, MAX_LEN
)
print(f'Train samples: {len(train_tag_ds)} | Test samples: {len(test_tag_ds)}')

In [ ]:
# ============================================================
# MODEL & TRAINER TAG
# ============================================================
NUM_LABELS_TAG = len(le_tag.classes_)
model_tag = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS_TAG
)

TAG_OUTPUT = OUTPUT_DIR + 'tag/'
os.makedirs(TAG_OUTPUT, exist_ok=True)

args_tag = TrainingArguments(
    output_dir=TAG_OUTPUT,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=WARMUP_RATIO,
    weight_decay=WEIGHT_DECAY,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1_macro',
    logging_steps=50,
    report_to='none',
    fp16=torch.cuda.is_available(),
    dataloader_num_workers=2
)

trainer_tag = Trainer(
    model=model_tag,
    args=args_tag,
    train_dataset=train_tag_ds,
    eval_dataset=test_tag_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print('🚀 Mulai fine-tuning Tag...')
trainer_tag.train()

In [ ]:
# ============================================================
# EVALUASI TAG
# ============================================================
y_pred_tag, y_true_tag = run_evaluation(
    trainer_tag, test_tag_ds, le_tag, 'Klasifikasi Tag'
)

trainer_tag.save_model(TAG_OUTPUT + 'best_model/')
print(f'\n✅ Model tag disimpan: {TAG_OUTPUT}best_model/')

## 7. Ringkasan Hasil & Simpan Prediksi

In [ ]:
# ============================================================
# RINGKASAN PERFORMA KESELURUHAN
# ============================================================
from sklearn.metrics import f1_score, accuracy_score

acc_sent  = accuracy_score(y_true_sent, y_pred_sent)
f1_sent   = f1_score(y_true_sent, y_pred_sent, average='macro')
acc_tag   = accuracy_score(y_true_tag, y_pred_tag)
f1_tag    = f1_score(y_true_tag, y_pred_tag, average='macro')

print('=' * 55)
print('          RINGKASAN EVALUASI FINAL')
print('=' * 55)
print(f'  Task Sentimen → Accuracy: {acc_sent:.3f} | F1-Macro: {f1_sent:.3f}')
print(f'  Task Tag      → Accuracy: {acc_tag:.3f} | F1-Macro: {f1_tag:.3f}')
print('=' * 55)

threshold_met = '✅' if f1_sent >= 0.85 and f1_tag >= 0.85 else '⚠️ Belum mencapai target 85%'
print(f'\n  Target F1 ≥ 0.85: {threshold_met}')
if f1_sent < 0.85 or f1_tag < 0.85:
    print('  → Pertimbangkan: tambah epoch, turunkan LR, atau tambah data augmentasi')

In [ ]:
# ============================================================
# SIMPAN PREDIKSI KE CSV
# ============================================================
df_result = df_test.copy()
df_result['pred_sentiment']     = le_sent.inverse_transform(y_pred_sent)
df_result['pred_tag']           = le_tag.inverse_transform(y_pred_tag)
df_result['correct_sentiment']  = df_result['Sentiment'] == df_result['pred_sentiment']
df_result['correct_tag']        = df_result['Tag'] == df_result['pred_tag']

RESULT_PATH = OUTPUT_DIR + 'test_predictions.csv'
df_result.to_csv(RESULT_PATH, index=False)
print(f'✅ Prediksi disimpan: {RESULT_PATH}')
print(f'\nContoh hasil prediksi:')
df_result[['Sentiment', 'pred_sentiment', 'correct_sentiment',
           'Tag', 'pred_tag', 'correct_tag']].head(10)

## 8. [BONUS] Training History Plot
Visualisasi loss dan F1 per epoch

In [ ]:
def plot_training_history(trainer, title):
    logs = trainer.state.log_history
    train_loss, eval_f1, epochs = [], [], []

    for log in logs:
        if 'loss' in log and 'epoch' in log:
            train_loss.append(log['loss'])
        if 'eval_f1_macro' in log:
            eval_f1.append(log['eval_f1_macro'])
            epochs.append(log['epoch'])

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(train_loss, marker='o', color='steelblue')
    ax1.set_title(f'{title} — Training Loss')
    ax1.set_xlabel('Step')
    ax1.set_ylabel('Loss')
    ax1.grid(True, alpha=0.3)

    ax2.plot(epochs, eval_f1, marker='o', color='green')
    ax2.axhline(y=0.85, color='red', linestyle='--', label='Target 85%')
    ax2.set_title(f'{title} — Eval F1-Macro per Epoch')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('F1-Macro')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR + f'history_{title.replace(" ","_")}.png', dpi=150)
    plt.show()

plot_training_history(trainer_sent, 'Sentimen')
plot_training_history(trainer_tag, 'Tag')

---
## ✅ Checklist Setelah Menjalankan Notebook Ini

| Item | Status |
|------|--------|
| Fine-tuning sentimen selesai | ☐ |
| Fine-tuning tag selesai | ☐ |
| F1-Macro sentimen ≥ 0.85 | ☐ |
| F1-Macro tag ≥ 0.85 | ☐ |
| Model disimpan ke Drive | ☐ |
| `test_predictions.csv` tersimpan | ☐ |

**Jika F1 < 0.85**, coba:
- Naikkan `EPOCHS` ke 7–10
- Turunkan `LR` ke `1e-5`
- Turunkan `MAX_LEN` ke 128 (lebih cepat, lebih banyak iterasi)
- Tambah data augmentasi untuk kelas yang performanya rendah